In [1]:
import math


def log_area_ratio(mach, gamma):
    """Logarithm of A/A* from Eq. (8–75)."""
    if mach == 0.0:
        return math.inf
    b = (gamma - 1.0) / 2.0
    exponent = (gamma + 1.0) / (2.0 * (gamma - 1.0))
    return exponent * (math.log1p(b * mach**2) - math.log1p(b)) - math.log(mach)


def static_pressure_ratio(mach, gamma):
    """Isentropic p/p0 at a given Mach number."""
    return math.exp(-gamma / (gamma - 1.0) * math.log1p((gamma - 1.0) * mach**2 / 2.0))


def normal_shock(mach_before, gamma):
    """Return the downstream Mach number and static pressure ratio p2/p1."""
    b = (gamma - 1.0) / 2.0
    m_squared = mach_before**2
    mach_after = math.sqrt((1.0 + b * m_squared) / (gamma * m_squared - b))
    pressure_jump = 1.0 + 2.0 * gamma / (gamma + 1.0) * (m_squared - 1.0)
    return mach_after, pressure_jump


def total_pressure_recovery(mach_before, gamma):
    mach_after, pressure_jump = normal_shock(mach_before, gamma)
    return (pressure_jump * static_pressure_ratio(mach_before, gamma)
            / static_pressure_ratio(mach_after, gamma))


def bisect_decreasing(function, target, lower, upper):
    """Invert a decreasing function over a bracket containing the root."""
    if not function(lower) >= target >= function(upper):
        raise ValueError("The requested Mach number is outside the model's numerical bounds.")
    for _ in range(100):
        midpoint = (lower + upper) / 2.0
        if midpoint == lower or midpoint == upper:
            break
        if function(midpoint) > target:
            lower = midpoint
        else:
            upper = midpoint
    return (lower + upper) / 2.0


def nozzle_limits(exit_area_ratio, gamma):
    """Return both isentropic exit Mach numbers and the three back-pressure limits."""
    target = math.log(exit_area_ratio)
    mach_sub = bisect_decreasing(lambda m: log_area_ratio(m, gamma), target, 0.0, 1.0)
    upper = 2.0
    while log_area_ratio(upper, gamma) < target:
        upper *= 2.0
    mach_sup = bisect_decreasing(lambda m: -log_area_ratio(m, gamma), -target, 1.0, upper)
    onset = static_pressure_ratio(mach_sub, gamma)
    design = static_pressure_ratio(mach_sup, gamma)
    _, pressure_jump = normal_shock(mach_sup, gamma)
    return mach_sub, mach_sup, onset, design * pressure_jump, design


def solve_nozzle(p0, pb, exit_area_ratio, gamma=1.4):
    if not all(math.isfinite(value) for value in (p0, pb, exit_area_ratio, gamma)):
        raise ValueError("All inputs must be finite.")
    if not (p0 > 0.0 and 0.0 <= pb <= p0 and exit_area_ratio > 1.0 and gamma > 1.0):
        raise ValueError("Require p0 > 0, 0 ≤ pb ≤ p0, Ae/At > 1, and gamma > 1.")

    mach_sub, mach_sup, onset, shock_at_exit, design = nozzle_limits(exit_area_ratio, gamma)
    ratio = pb / p0
    # Keep each endpoint tolerance smaller than the gap to the adjacent regime.
    tolerance = min(1e-10, (onset - shock_at_exit) / 4.0, (shock_at_exit - design) / 4.0)
    if tolerance <= 0.0:
        raise ValueError("These inputs do not give three distinguishable pressure limits.")

    result = {"onset": onset, "shock_at_exit": shock_at_exit, "design": design}
    if pb == p0:
        result.update(regime="No flow (equal reservoir and back pressures)", exit_mach=0.0)
    elif ratio > onset + tolerance:
        exponent = -(gamma - 1.0) / gamma * math.log(ratio)
        exit_mach = math.sqrt(2.0 * math.expm1(exponent) / (gamma - 1.0))
        result.update(regime="Unchoked subsonic flow", exit_mach=exit_mach)
    elif abs(ratio - onset) <= tolerance:
        result.update(regime="Onset of choking (sonic throat, subsonic exit)", exit_mach=mach_sub)
    elif ratio > shock_at_exit + tolerance:
        # Eq. (8–86), rationalized to avoid subtracting nearly equal numbers.
        b = (gamma - 1.0) / 2.0
        mass_flow_factor = (1.0 + b) ** (-(gamma + 1.0) / (2.0 * (gamma - 1.0)))
        s = (mass_flow_factor / (ratio * exit_area_ratio)) ** 2
        exit_mach = math.sqrt(2.0 * s / (1.0 + math.sqrt(1.0 + 4.0 * b * s)))
        recovery = ratio / static_pressure_ratio(exit_mach, gamma)
        mach_before = bisect_decreasing(lambda m: total_pressure_recovery(m, gamma), recovery, 1.0, mach_sup)
        mach_after, _ = normal_shock(mach_before, gamma)
        shock_area_ratio = math.exp(log_area_ratio(mach_before, gamma))
        if not 1.0 < shock_area_ratio < exit_area_ratio:
            raise ValueError("Computed shock position lies outside the divergent section.")
        result.update(regime="Normal shock inside the divergent section", exit_mach=exit_mach,
                      shock_area_ratio=shock_area_ratio, mach_before=mach_before,
                      mach_after=mach_after, recovery=recovery)
    elif abs(ratio - shock_at_exit) <= tolerance:
        mach_after, _ = normal_shock(mach_sup, gamma)
        result.update(regime="Normal shock at the exit plane", shock_area_ratio=exit_area_ratio,
                      mach_before=mach_sup, mach_after=mach_after,
                      recovery=total_pressure_recovery(mach_sup, gamma))
    else:
        if ratio > design + tolerance:
            regime = "Overexpanded jet (compression outside the nozzle)"
        elif abs(ratio - design) <= tolerance:
            regime = "Perfectly expanded supersonic flow"
        else:
            regime = "Underexpanded jet (expansion outside the nozzle)"
        result.update(regime=regime, exit_mach=mach_sup)
    return result


def read_value(label, default, valid, requirement):
    while True:
        raw = input(f"{label} [{default:g}]: ").strip()
        try:
            value = float(raw) if raw else default
        except ValueError:
            print("Enter a number or press Enter to use the default.")
            continue
        if math.isfinite(value) and valid(value):
            return value
        print(requirement)


def main():
    print("Chapter 8 — Example 8–4 defaults. Press Enter to accept each value.")
    p0 = read_value("Reservoir stagnation pressure p0 (kPa)", 101.32,
                    lambda p: p > 0.0, "Enter a positive reservoir pressure.")
    pb = read_value("Back pressure pb (kPa)", 0.5 * p0,
                    lambda p: 0.0 <= p <= p0, "Enter a back pressure between zero and p0.")
    exit_area_ratio = read_value("Exit-to-throat area ratio Ae/At", 3.0,
                                 lambda a: a > 1.0, "Enter an area ratio greater than 1.")
    gamma = read_value("Specific-heat ratio gamma", 1.4,
                       lambda g: g > 1.0, "Enter a specific-heat ratio greater than 1.")

    try:
        result = solve_nozzle(p0, pb, exit_area_ratio, gamma)
    except (ValueError, OverflowError) as error:
        print(f"The calculation could not be completed: {error}")
        return

    print(f"\nApplied back-pressure ratio pb/p0 = {pb / p0:.6g}")
    print("Back-pressure limits (ratio and absolute pressure)")
    for key, label in (("onset", "Onset of choking"),
                       ("shock_at_exit", "Normal shock at exit"),
                       ("design", "Perfect expansion")):
        limit = result[key]
        print(f"  {label:22s} {limit:.10g}   ({p0 * limit:.10g} kPa)")
    print(f"\nFlow regime: {result['regime']}")
    if "exit_mach" in result:
        print(f"Exit Mach number Me = {result['exit_mach']:.6g}")
    if "shock_area_ratio" in result:
        print(f"Shock location As/At = {result['shock_area_ratio']:.6g}")
        print(f"Mach immediately before shock = {result['mach_before']:.6g}")
        print(f"Mach immediately after shock  = {result['mach_after']:.6g}")
        print(f"Stagnation-pressure ratio p02/p01 = {result['recovery']:.6g}")


if __name__ == "__main__":
    main()


Chapter 8 — Example 8–4 defaults. Press Enter to accept each value.
Reservoir stagnation pressure p0 (kPa) [101.32]: 
Back pressure pb (kPa) [50.66]: 
Exit-to-throat area ratio Ae/At [3]: 
Specific-heat ratio gamma [1.4]: 

Applied back-pressure ratio pb/p0 = 0.5
Back-pressure limits (ratio and absolute pressure)
  Onset of choking       0.9731817988   (98.60277986 kPa)
  Normal shock at exit   0.375959457   (38.09221219 kPa)
  Perfect expansion      0.04729869168   (4.792303441 kPa)

Flow regime: Normal shock inside the divergent section
Exit Mach number Me = 0.38034
Shock location As/At = 2.33973
Mach immediately before shock = 2.37094
Mach immediately after shock  = 0.526272
Stagnation-pressure ratio p02/p01 = 0.552488
